# Screen Watcher Pro — Jupyter AI Chatbox (spec §12, FR06/T06)

Local web client cho **Tool Watcher API Server**. Client **không chứa logic AI** — chỉ thu thập input,
gửi HTTP request đến server và hiển thị response (spec §12.1).

**Trước khi chạy:** khởi động server trong một terminal:
```
uvicorn app.ai.chat_server:app --host 127.0.0.1 --port 8000 --workers 1
```
(Đặt `ai.mock: true` trong `config/rules.yaml` để demo không cần provider thật,
hoặc `ai.engine: opencode` để đi qua OpenCode CLI — spec §11.)

In [1]:
# --- 1. Setup: make `app` importable, configure the client ---
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import requests
from app.ai import chatbox

BASE_URL = "http://127.0.0.1:8000"
USERNAME = "admin"          # demo credentials (seeded admin account)
PASSWORD = "admin123"
SESSION_ID = None            # None -> the server creates a new conversation
print("Client configured for", BASE_URL)

Client configured for http://127.0.0.1:8000


## 2. Health check (FR01, demo scenario §18.1-1)
Server phải trả `status: ok` trước khi demo.

In [2]:
try:
    r = requests.get(f"{BASE_URL}/health", timeout=10)
    print("HTTP", r.status_code, "->", r.json())
except requests.ConnectionError:
    print("⚠ Server chưa chạy — hãy khởi động uvicorn (xem cell đầu tiên).")

HTTP 200 -> {'status': 'ok', 'provider': 'engine=sdk provider=openrouter model=openai/gpt-4o-mini timeout=120s max_ctx=6000 key[OPENROUTER_API_KEY]=mock'}


## 3. Đăng nhập lấy JWT token
API yêu cầu Bearer token (bảo mật hơn spec gốc — spec §16 khuyến nghị auth khi mở rộng).

In [3]:
TOKEN = chatbox.login(BASE_URL, USERNAME, PASSWORD)
print("Đăng nhập OK — token:", TOKEN[:24] + "…")

Đăng nhập OK — token: eyJhbGciOiJIUzI1NiIsInR5…


## 4. Chatbox widget (spec §12.2)
Chat UI bằng `ipywidgets`: history + ô nhập + nút **Send** + toggle *Include latest watcher context*.
Nếu ipywidgets lỗi, tự fallback sang vòng lặp `input()` (spec §19: notebook widget risk mitigation).

In [4]:
chatbox.launch_chatbox(BASE_URL, token=TOKEN)

## 5. Scripted demo — hỏi trạng thái watcher (demo scenario §18.1-2/4/5)
Gọi thẳng `send_message` (cùng hàm chatbox dùng) để có output lưu lại được trong notebook.

In [5]:
import uuid
sid = str(uuid.uuid4())   # server requires a UUID session id
reply = chatbox.send_message(BASE_URL, sid, "Trạng thái watcher gần nhất là gì?",
                             token=TOKEN, include_context=True)
print("You: Trạng thái watcher gần nhất là gì?")
print("Bot:", reply)
print("---")

You: Trạng thái watcher gần nhất là gì?
Bot: [MOCK mode] The assistant is running without a real LLM (ai.mock=true). Set ai.mock=false and configure the provider API key in .env to enable it.
---


## 6. Latest watcher result (FR07, demo scenario §18.1-3)
`GET /api/watcher/executions/latest` — execution id, OCR text, matched rules, email decisions.

In [6]:
r = requests.get(f"{BASE_URL}/api/watcher/executions/latest",
                 headers={"Authorization": f"Bearer {TOKEN}"}, timeout=30)
data = r.json()
print("HTTP", r.status_code)
print("has_data      :", data.get("has_data"))
print("execution_id  :", data.get("execution_id"))
print("target_app    :", data.get("target_app"))
print("captured_at   :", data.get("captured_at"))
print("matched_rules :", data.get("matched_rules"))
print("ocr_text      :", (data.get("ocr_text") or "")[:300])

HTTP 200
has_data      : True
execution_id  : 6cf3c4d6-4efb-4fbd-a00d-06814199fdc6
target_app    : chrome
captured_at   : 2026-07-03 02:05:06
matched_rules : [{'rule_id': 'daily_sync_failed', 'rule_name': 'Daily sync failed', 'severity': 'high', 'owner_group': 'ops_team', 'reason': "keyword 'FAILED' found in OCR text"}]
ocr_text      : Daily Sync Job
Status: FAILED
Error: connection timeout to payment-gateway
Last success: 02:00 AM


## 7. Trigger watcher thủ công (FR08 — optional)
Chạy capture+OCR+rule ngay (cần cửa sổ Chrome/Edge trên máy chạy server). Bỏ comment để dùng:

In [7]:
# r = requests.post(f"{BASE_URL}/api/watcher/executions",
#                   json={"targets": ["chrome"], "launch": False},
#                   headers={"Authorization": f"Bearer {TOKEN}"}, timeout=180)
# print(r.status_code, r.json())